In [ ]:
from pykrx import stock
import datetime
import pandas as pd
import math
from tqdm import tqdm
from gensim.models import word2vec
import random
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
import numpy as np
import plotly.express as px
from statsmodels.graphics.mosaicplot import mosaic
matplotlib.rcParams['axes.unicode_minus'] = False
from matplotlib import rc 
%matplotlib inline
rc('font', family='Malgun Gothic')

In [ ]:
data_1 = pd.read_csv('data/LPOINT_BIG_COMP_01_DEMO.csv')
data_2 = pd.read_csv('data/LPOINT_BIG_COMP_02_PDDE.csv')
data_3 = pd.read_csv('data/LPOINT_BIG_COMP_03_COP_U.csv')
data_4 = pd.read_csv('data/LPOINT_BIG_COMP_04_PD_CLAC.csv')
data_5 = pd.read_csv('data/LPOINT_BIG_COMP_05_BR.csv')
data_6 = pd.read_csv('data/LPOINT_BIG_COMP_06_LPAY.csv')

## 전처리

### data_6

In [ ]:
data_61=data_6.groupby("cust")["cop_c"].agg(**{'most_cop':lambda x:x.mode()[0]})
data_62=data_6.groupby("cust")["chnl_dv"].agg(**{'most_chnl':lambda x:x.mode()[0]})
data_63=data_6.groupby("cust")["buy_am"].agg(**{'point_sum':"sum"})
data_64=data_6.groupby("cust")["buy_am"].agg(**{'point_count':"count"})

In [ ]:
data_6_f = pd.concat([data_61, data_62, data_63, data_64], axis=1).reset_index()

In [ ]:
data_6_f.head()

### data_2,3 결측치 처리
- br_c NaN값은 온라인이므로 Nan값을 온라인으로 처리

In [ ]:
data_2 = data_2.fillna('온라인')
data_3 = data_3.fillna('온라인')

### data_1,4,5는 마스터 테이블 개념으로 다른 데이터에 병합가능

#### 1) data_2에 pd_c를 기준으로 data_4에 있는 정보를 추가

In [ ]:
data_2 = pd.merge(data_2, data_4, how='left',on='pd_c')
data_2.head()

#### 2-1) data_2에 br_c 기준으로 data_5에 있는 정보를 추가

In [ ]:
data_2 = pd.merge(data_2, data_5[data_5.columns.difference(['cop_c'])], how='left',on='br_c')
data_2.head()

#### 2-2) data_3에 br_c 기준으로 data_5에 있는 정보를 추가

In [ ]:
data_3 = pd.merge(data_3, data_5[data_5.columns.difference(['cop_c'])], how='left',on='br_c')
data_3.head()

#### 3-1) data_2에 cust 기준으로 data_1에 있는 정보를 추가

In [ ]:
data_2 = pd.merge(data_2.drop(columns='zon_hlv'), data_1, how='left',on='cust')
data_2.head()

#### 3-2) data_3에 cust 기준으로 data_1에 있는 정보를 추가

In [ ]:
data_3 = pd.merge(data_3.drop(columns='zon_hlv'), data_1, how='left',on='cust')
data_3.head()

#### 3-3) data_6에 cust 기준으로 data_1에 있는 정보를 추가

In [ ]:
data_6 = pd.merge(data_6, data_1, how='left',on='cust')
data_6.head()

### data_2,3,6에 data_1,4,5정보를 넣었으므로 data_2,3,6으로만 진행

In [ ]:
data_2 = data_2.fillna('없음')
data_3 = data_3.fillna('없음')

### 2번데이터 중복값 합치기
- 중복데이터는 같은상품을 여러번 구매를 했거나 하는 등 중복된 건은 중복된 이유가 있습니다. 
- 완전히 상품단위로 제공되는 데이터가 아니기 때문에 일정 카테고리 단위로 데이터를 통일시킨 거라 중복된 건은 원래 중복된 데이터가맞습니다.
- 따라서 중복데이터를 그냥 제거하는것이 아닌 buy_am, buy_ct에는 중복된 값을 더하고 나머지는 제거

In [ ]:
data_2["dup"] = np.ones(len(data_2))
dup = data_2.groupby(['cust', 'rct_no', 'chnl_dv', 'cop_c', 'br_c', 'pd_c', 'de_dt', 'de_hr',
       'buy_am', 'buy_ct', 'pd_nm', 'clac_hlv_nm', 'clac_mcls_nm', 'zon_mcls',
       'ma_fem_dv', 'ages', 'zon_hlv'])["dup"].sum()
dup = pd.DataFrame(dup).reset_index()
data_2 = pd.merge(data_2, dup, on = ['cust', 'rct_no', 'chnl_dv', 'cop_c', 'br_c', 'pd_c', 'de_dt', 'de_hr',
       'buy_am', 'buy_ct', 'pd_nm', 'clac_hlv_nm', 'clac_mcls_nm', 'zon_mcls',
       'ma_fem_dv', 'ages', 'zon_hlv'])
data_2["buy_am"] = data_2["buy_am"]*data_2["dup_y"]
data_2["buy_ct"] = data_2["buy_ct"]*data_2["dup_y"]
del data_2["dup_x"]
data_2.rename(columns={"dup_y":"중복수"}, inplace=True)
data_2 = data_2.drop_duplicates()
data_2 = data_2.reset_index(drop = True)

In [ ]:
np.shape(data_2)

In [ ]:
data_2.head()

## 지역별

- 지역 데이터를 활용하여 수도권, 비수도권 구분

In [ ]:
pd.DataFrame(data_2.groupby('zon_hlv').cust.count()).sort_values('cust', ascending=False).head()

In [ ]:
pd.DataFrame(data_3.groupby('zon_hlv').cust.count()).sort_values('cust', ascending=False).head()

In [ ]:
pd.DataFrame(data_6.groupby('zon_hlv').cust.count()).sort_values('cust', ascending=False).head()

In [ ]:
data_2_eda = data_2.copy()
Metro_lst = ['Z10', 'Z17']
data_2_eda["zon_hlv"]=data_2_eda["zon_hlv"].apply(lambda x: '수도권' if x in Metro_lst else '비수도권')
data_2_eda = pd.DataFrame(data_2_eda.groupby('zon_hlv')['buy_am'].sum()/data_2_eda.groupby('zon_hlv')['buy_ct'].sum())
data_2["city"]=data_2["zon_hlv"].apply(lambda x: 1 if x in Metro_lst else 0)
data_2_eda.columns = ['평균구매금액']
data_2_eda

In [ ]:
df3 = data_2_eda.평균구매금액
f,ax=plt.subplots(1,2,figsize=(18,8))
df3.plot.pie(explode=[0,0.1],autopct='%1.1f%%',ax=ax[0],shadow=True)
ax[0].set_title('평균구매금액')
ax[0].set_ylabel('')
data_2_eda['평균구매금액'].plot(kind='bar', rot=0)
ax[1].set_title('평균구매금액')
plt.show()

In [ ]:
del data_2_eda

In [ ]:
Metro_lst=["Z17", "Z10"]

In [ ]:
data_2["city"]=data_2["zon_hlv"].apply(lambda x: 1 if x in Metro_lst else 0)

#### Z17, Z10은 경기도 및 서울일 가능성이 높음 / Z02, Z07은 세종,제주일 가능성이 높음
#### Z17 경기, Z10 서울 Z16 부산으로 예상 Z02는 세종, Z07은 제주로 예상
#### Z17,Z10은 수도권이 확실하므로 외부 인구수 공공데이터를 활용하여 수도권 비수도권여부 피처 생성

## 공휴일

- 2021년 법정 공휴일 구분

In [ ]:
import holidays
start_date='20210101'
end_date='20211231'
date_list=pd.date_range(start=start_date, end=end_date, freq='D')

In [ ]:
kr_holidays = holidays.KR()

In [ ]:
holiday_df = pd.DataFrame(columns=['de_dt','holiday'])
holiday_df['de_dt'] = sorted(date_list)
holiday_df['holiday'] = holiday_df.de_dt.apply(lambda x: 1 if x in kr_holidays else 0)

In [ ]:
data_2["de_dt"]=pd.to_datetime(data_2["de_dt"].astype("str"))

In [ ]:
holiday_df.head()
data_2=pd.merge(data_2, holiday_df, how="left")

In [ ]:
data_2_holi = data_2.copy()
data_2_holi = pd.DataFrame(data_2_holi.groupby('holiday')['buy_am'].sum()/data_2_holi.groupby('holiday')['buy_ct'].sum())
data_2_holi.set_index([['비공휴일','공휴일']], inplace=True)
data_2_holi.columns = ['평균구매금액']
data_2_holi

In [ ]:
df3 = data_2_holi.평균구매금액
f,ax=plt.subplots(1,2,figsize=(18,8))
df3.plot.pie(explode=[0,0.1],autopct='%1.1f%%',ax=ax[0],shadow=True)
ax[0].set_title('평균구매금액')
ax[0].set_ylabel('')
data_2_holi['평균구매금액'].plot(kind='bar', rot=0)
ax[1].set_title('평균구매금액')
plt.show()

## 주식 데이터 이용, 출처 : 네이버 금융

- 날짜 별 코스피 지수 및 롯데 지주 주가

In [ ]:
kospi=stock.get_index_fundamental("20201230", "20211231","1001")[["종가", "등락률"]].reset_index()
lotte=stock.get_market_ohlcv_by_date("20201230", "20211231","004990")[["종가", "거래량"]].reset_index()

In [ ]:
kospi.rename(columns={"날짜":"de_dt","종가":"종가_kospi","등락률":"등락률_kospi"}, inplace=True)
lotte.rename(columns={"날짜":"de_dt", "종가":"종가_lotte","거래량":"거래량_lotte"}, inplace=True)

In [ ]:
kospi["de_dt"]=kospi["de_dt"].apply(lambda x: x + datetime.timedelta(days=1))
lotte["de_dt"]=kospi["de_dt"].apply(lambda x: x + datetime.timedelta(days=1))

In [ ]:
start_date='20201230'
end_date='20211231'
date_list=pd.date_range(start=start_date, end=end_date, freq='D')
jusik=pd.DataFrame({"de_dt":date_list})
jusik=pd.merge(pd.merge(jusik,kospi,how="left"),lotte,how="left")

In [ ]:
jusik["등락률_kospi"].fillna(0,inplace=True)
jusik.fillna(method="ffill",inplace=True)
jusik=jusik.iloc[2:,:]
jusik.head()

In [ ]:
data_2=pd.merge(data_2, jusik, on="de_dt",  how="left")
data_2.head()

In [ ]:
df_buy_ct = pd.DataFrame(data_2.groupby('de_dt')['cust'].nunique())
df_buy_am = pd.DataFrame(data_2.groupby('de_dt')['buy_am'].sum())
df_buy_am = df_buy_am.buy_am / df_buy_ct.cust
df_buy = pd.concat([df_buy_ct, df_buy_am], axis = 1)
df_buy.columns = ['고객수', '평균구매금액']

In [ ]:
jusik = jusik.reset_index(drop = True)
df_buy = df_buy.reset_index(drop = True)
jusik_buy = pd.concat([jusik, df_buy], axis = 1)

In [ ]:
for jong in ["종가_kospi", "등락률_kospi", "종가_lotte"]:
    sns.jointplot(jong, "고객수", height=6, data=jusik_buy, kind="reg", color ="deepskyblue")
    plt.title(jong + "에 따른 고객수 변화")
    plt.show()

In [ ]:
jusik_buy=jusik_buy.query("거래량_lotte<1000000")

In [ ]:
for jong in ["거래량_lotte"]:
    sns.jointplot(jong, "고객수", height=6, data=jusik_buy, kind="reg", color ="deepskyblue")
    plt.title(jong + "에 따른 고객수 변화")
    plt.show()

## 날씨 데이터 출처 : 기상청

- 날짜 별 기온 및 강수량

In [ ]:
temp=pd.read_csv("data/external/기상청_2021일자별 기온(출처_기상청_기상자료개방포털.csv", encoding="euc-kr")
precp=pd.read_csv("data/external/기상청_2021일자별_강수량(출처_기상청_기상자료개방포털).csv", encoding="euc-kr")

In [ ]:
gisang=pd.merge(temp[["날짜","평균기온(℃)", "최저기온(℃)", "최고기온(℃)"]],precp[["날짜","강수량(mm)"]])
gisang

In [ ]:
gisang["날짜"]=pd.to_datetime(gisang["날짜"].astype("str"))
gisang.rename(columns={"날짜":"de_dt"}, inplace=True)
data_2=pd.merge(data_2,gisang,on="de_dt", how="left")

## 문제 정의 시각화 및 외부 데이터 시각화

- 문제 정의

In [ ]:
data_2["month"]=data_2["de_dt"].dt.month

In [ ]:
nu_m=pd.DataFrame(data_2.groupby("cust")["month"].nunique())

In [ ]:
data_2

In [ ]:
plt.figure(figsize=(8,4))
ax=sns.countplot(x="month", data=nu_m)
ax.set_xticklabels(ax.get_xticklabels(),  fontsize=13)
plt.tight_layout()
plt.title("고객 별 오는 월 수")
plt.show()

In [ ]:
nu_m["m_group"]=nu_m["month"].apply(lambda x: "A" if x < 4 else("B" if x< 10 else "C"))

In [ ]:
hi=nu_m.groupby("m_group")["month"].count().reset_index()
group_name = hi[hi.columns[0]]
group_size = hi[hi.columns[1]]
hi

In [ ]:
bye=nu_m.groupby("month")["m_group"].count().reset_index()
bye["group"]=["A"]*3+["B"]*6+["C"]*3
subgroup_name = bye[bye.columns[0]]
subgroup_size = bye[bye.columns[1]]
tmp_subgroup_name = bye[bye.columns[2]]
bye

In [ ]:
zero_box = np.zeros(len(group_name))
for i in range(len(group_name)):
    for j in tmp_subgroup_name:
        if j == group_name[i]:
            zero_box[i] += 1
        else:
            pass

In [ ]:
import random
color_list = np.zeros([len(group_name), 4])
color_list[:, 3] = 1

for i in range(len(group_name)):
    color_list[i][0] = random.random()
    color_list[i][1] = random.random()
    color_list[i][2] = random.random()

In [ ]:
sub_size = 0
for i in zero_box:
    sub_size += i
    
sub_color_list = np.zeros([int(sub_size), 4])

order = 0 # 리스트
order2 = 0 # 
for i in zero_box:
    if i == 1:
        color_range = 1
        sub_color_list[order][0] = color_list[order2][0]
        sub_color_list[order][1] = color_list[order2][1]
        sub_color_list[order][2] = color_list[order2][2]
        sub_color_list[order][3] = 1
        order+=1
        
    else:
        color_range = np.linspace(0.1, 1, int(i))
        for j in range(int(i)):
            sub_color_list[order][0] = color_list[order2][0]
            sub_color_list[order][1] = color_list[order2][1]
            sub_color_list[order][2] = color_list[order2][2]
            sub_color_list[order][3] = color_range[j]
            order+=1
    order2+=1

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))

ax.axis('equal')

pie_outside, _ = ax.pie(group_size, radius=1.3, 
                        labels=group_name, 
                        labeldistance=0.8,
                        colors=color_list)
plt.setp(pie_outside, 
         width=0.5, 
         edgecolor='white')
# Inside Ring

pie_inside, plt_labels, junk = \
    ax.pie(subgroup_size, 
           radius=(1.3 - 0.5), 
           labels=subgroup_name, 
           labeldistance=0.75, 
           autopct='%1.f%%',
           colors=sub_color_list)
plt.setp(pie_inside, 
         width=0.5, 
         edgecolor='white')
plt.title('', fontsize=20)
plt.show()

In [ ]:
first=list(data_2.query('month==1').cust.unique())

In [ ]:
c_c=[]
for i in range(1,13):
    c_c.append(len(set(data_2.query('month==@i').cust.unique()) & set(first)))
    first=set(data_2.query('month==@i').cust.unique()) & set(first)

In [ ]:
sns.set(style='darkgrid')
plt.bar(['1','2','3','4','5','6','7','8','9','10','11','12'],c_c,
       color=["royalblue"]+['skyblue']*11, edgecolor="gray")
plt.show()

In [ ]:
del data_2["month"]

- 외부 데이터 특징 확인

In [ ]:
opop=data_2.groupby("de_dt").agg({"cust":"nunique","평균기온(℃)":"mean", "최저기온(℃)":"mean", "최고기온(℃)":"mean", "강수량(mm)":"mean"})

In [ ]:
data_2_eda = data_2.copy()
Metro_lst = ['Z10', 'Z17']
data_2_eda["zon_hlv"]=data_2_eda["zon_hlv"].apply(lambda x: '수도권' if x in Metro_lst else '비수도권')
data_2_eda = pd.DataFrame(data_2_eda.groupby('zon_hlv')['buy_am'].sum()/data_2_eda.groupby('zon_hlv')['buy_ct'].sum())
data_2["city"]=data_2["zon_hlv"].apply(lambda x: 1 if x in Metro_lst else 0)
data_2_eda.columns = ['평균구매금액']

In [ ]:
opop["일교차"]=opop["최고기온(℃)"]-opop["최저기온(℃)"]

In [ ]:
data_2_eda

In [ ]:
opop.columns[1:].to_list()

In [ ]:
for a in opop.columns[1:].to_list():
    olol=opop.groupby(a)["cust"].mean().reset_index()
    poly = np.polyfit(olol[a], olol["cust"], 5) # 5차, 4차,.. 상수항 계수가 순서대로 들어있는 배열
    x_line = np.linspace(olol[a].min()+1, olol[a].max()-1, 1000) # 곡선을 완만하게 그리기 위한 linspace 객체 선언
    y_pred = x_line ** 5 * poly[0] + x_line ** 4 * poly[1] + x_line ** 3 * poly[2] + x_line ** 2 * poly[3] + x_line ** 1 * poly[4] + poly[5]
    plt.title(str(a) +"별 고객 방문 수")
    plt.plot(x_line, y_pred, color = 'royalblue')
    plt.show()

### 달 설정

In [ ]:
train_mt_prev=int('2021'+input("data 이용 기간 설정  입력 예시   3월 이후 -> 03    : ")+'01')
train_mt_after=int('2021'+input("data 이용 기간 설정  입력 예시   9월 이전 -> 09      :")+'31')
data_2=data_2.query("de_dt<=@train_mt_after & de_dt>=@train_mt_prev")
data_3=data_3.query("de_dt<=@train_mt_after & de_dt>=@train_mt_prev")
data_6=data_6.query("de_dt<=@train_mt_after & de_dt>=@train_mt_prev")

### 고객 별 선호하는 제품 및 제휴사 고객 별 특징 파악 - 주 선호하는 구매시간, 이용시간, 물품 종류, 제휴사, 점포

In [ ]:
features = []

- 총구매액, 구매건수, 평균구매액, 최대구매액, 최소구매액

In [ ]:
f = data_2.groupby('cust')['buy_am'].agg([('총구매액', np.sum),
                                            ('구매건수', np.size),
                                            ('평균구매액', lambda x : np.round(np.mean(x))),
                                            ('최대구매액', np.max),
                                            ('최소구매액', np.min)]).reset_index()

features.append(f)

- 주말방문비율, 주중방문비율

In [ ]:
data_2['de_dt']= data_2['de_dt'].astype('str')
data_2['de_dt'] = pd.to_datetime(data_2['de_dt'])

In [ ]:
data_2['sales_dayofweek'] = data_2['de_dt'].dt.day_name()

day = {'Monday' : 0 , 'Tuesday' : 1 , 'Wednesday' : 2 , 'Thursday' : 3 , 'Friday' : 4 , 'Saturday': 5 , 'Sunday' : 6 }

data_2['sales_dayofweek_num'] = data_2['sales_dayofweek'].apply(lambda x : day[x] )

f = data_2.groupby('cust')['sales_dayofweek_num'].agg([
    ('주말방문비율', lambda x: np.mean(x >4)),
    ('주중방문비율' , lambda x : np.mean(x<5))
]).reset_index()

features.append(f)

- 계절방문비율

In [ ]:
data_2['sales_month'] = data_2['de_dt'].dt.month
#
#f = data_2.groupby('cust')['sales_month'].agg([
#    ('봄-구매비율', lambda x: np.mean( x.isin([3,4,5]))),
#    ('여름-구매비율', lambda x: np.mean( x.isin([6,7,8]))),
#    ('가을-구매비율', lambda x: np.mean(x.isin([9,10,11]))),
#    ('겨울-구매비율', lambda x: np.mean( x.isin([1,2,12])))
#]).reset_index()
#
#features.append(f)

- [월별] 총구매액, 구매건수, 평균구매액, 최대구매액, 최소구매액

In [ ]:
#f = pd.pivot_table(data_2,index='cust',columns='sales_month',values='buy_am',aggfunc = ['sum','size','mean','max','min'],fill_value=0).reset_index()

#features.append(f)

- 시간대별방문비율 : 아침, 점심, 저녁으로 구분
- 0-5시는 새벽, 5-9시는 아침, 9-17시는 낮, 17-21시는 저녁, 21-24시는 밤

In [ ]:
f = data_2.groupby('cust')['de_hr'].agg([
    ('새벽-구매비율', lambda x: np.mean( x.isin([0,1,2,3,4]))),
    ('아침-구매비율', lambda x: np.mean( x.isin([5,6,7,8]))),
    ('낮-구매비율', lambda x: np.mean( x.isin([9,10,11,12,13,14,15,16]))),
    ('저녁-구매비율', lambda x: np.mean( x.isin([17,18,19,20]))),
    ('밤-구매비율', lambda x: np.mean(x.isin([21,22,23])))
]).reset_index()

features.append(f)

- 온,오프라인 구매 비율(1: 오프라인, 2: 온라인)

In [ ]:
f = data_2.groupby('cust')['chnl_dv'].agg([
    ('오프라인구매비율', lambda x: np.mean( x==1)),
    ('온라인구매비율', lambda x: np.mean( x==2))
]).reset_index()

features.append(f)

- 내점일수, 구매주기

In [ ]:
data_2['sales_day'] = data_2['de_dt'].dt.day

In [ ]:
data_2['sales_date'] =data_2['sales_month'].astype(str).apply(lambda x : "0"+x if len(x) == 1  else x ) +data_2['sales_day'].astype(str).apply(lambda x : "0"+x if len(x) == 1 else  x )
data_2['sales_date'] = data_2['sales_date'].astype(int)

f = data_2.groupby('cust')['sales_date'].agg([
    ('내점일수',lambda x: x.nunique()),
    ('구매주기', lambda x: (x.max() - x.min()) / x.nunique())]).reset_index()

features.append(f)

- 주방문요일

In [ ]:
f = data_2.groupby('cust')['sales_dayofweek'].agg([('주방문요일', lambda x: x.mode()[0])]).reset_index()
f = pd.get_dummies(f, columns=['주방문요일'])

features.append(f)

- 평균실구매액

In [ ]:
data_2['real_amt'] = ( data_2['buy_am'] / data_2['buy_ct'] ).apply(lambda x : math.trunc(x)) 

f = data_2.groupby('cust')['real_amt'].agg([('평균실구매액', 'mean')]).reset_index()

features.append(f)

- 고가상품구매비율, 저가상품구매비율

In [ ]:
buy_90 = data_2.buy_am.quantile(.90)
buy_10 = data_2.buy_am.quantile(.10)

f = data_2.groupby('cust')['buy_am'].agg([
    ('고가상품구매비율', lambda x: np.mean(x > buy_90 )),
    ('저가상품구매비율', lambda x: np.mean( x < buy_10 ))
]).reset_index()

features.append(f)

- 주구매지역

In [ ]:
f = data_2.groupby('cust')['zon_hlv'].agg([('주구매지역', lambda x: x.value_counts().index[0])]).reset_index()
f = pd.get_dummies(f, columns=['주구매지역'])

features.append(f) ;f.head()

- 주구매제휴사

In [ ]:
f = data_2.groupby('cust')['cop_c'].agg([('주구매제휴사', lambda x: x.value_counts().index[0])]).reset_index()
f = pd.get_dummies(f, columns=['주구매제휴사'])

features.append(f) ;f.head()

- 주구매지역

In [ ]:
f = data_2.groupby('cust')['zon_hlv'].agg([('주구매지역', lambda x: x.value_counts().index[0])]).reset_index()
f = pd.get_dummies(f, columns=['주구매지역'])

features.append(f) ;f.head()

- 주구매제휴사

In [ ]:
f = data_2.groupby('cust')['cop_c'].agg([('주구매제휴사', lambda x: x.value_counts().index[0])]).reset_index()
f = pd.get_dummies(f, columns=['주구매제휴사'])

features.append(f) ;f.head()

- 평균구매시간 , 총구매시간

In [ ]:
data_2['total_sec'] = data_2['de_hr'].apply(lambda x : x//100*60 + x%100 )

In [ ]:
f = data_2.groupby('cust')['total_sec'].agg([
    ('평균쇼핑시간', lambda x: (x.max() - x.min()) / x.nunique()),
    ('총쇼핑시간' , 'sum')]).reset_index()

features.append(f) ;f.head()

- 구매요일다양성

In [ ]:
a = data_2.sales_dayofweek.nunique()
f = data_2.groupby('cust')['sales_dayofweek'].agg([('구매요일다양성', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f) ;f.head()

- 구매제휴사다양성

In [ ]:
a = data_2.cop_c.nunique()
f = data_2.groupby('cust')['cop_c'].agg([('구매제휴사다양성', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f); f.head()

- 구매점포다양성

In [ ]:
a = data_2.br_c.nunique()
f = data_2.groupby('cust')['br_c'].agg([('구매점포다양성', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f); f.head()

- 구매지역다양성

In [ ]:
a = data_2.zon_hlv.nunique()
f = data_2.groupby('cust')['zon_hlv'].agg([('구매지역다양성', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f) ;f.head()

- 구매상품소분류다양성

In [ ]:
a = data_2.pd_nm.nunique()
f = data_2.groupby('cust')['pd_nm'].agg([('구매상품소분류', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f) ;f.head()

- 주구매상품중분류다양성

In [ ]:
a = data_2.clac_hlv_nm.nunique()
f = data_2.groupby('cust')['clac_hlv_nm'].agg([('주구매상품중분류', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f) ;f.head()

- 주구매상품대분류다양성

In [ ]:
a = data_2.clac_mcls_nm.nunique()
f = data_2.groupby('cust')['clac_mcls_nm'].agg([('구매지역다양성', lambda x: len(x.unique()) / a)]).reset_index()
features.append(f); f.head()

- 이벤트성 고객 (총 구매횟수가 1번인 고객)

In [ ]:
f = data_2.groupby('cust')['de_dt'].agg([('n_date', 'nunique')]).reset_index()
f['one_visit'] = f['n_date'].apply(lambda x: 1 if x<2 else 0)

features.append(f) ;f.head()

In [ ]:
len(features)

In [ ]:
data = features[0]
for f in tqdm(features[1:]):
    data = pd.merge(data, f, on = 'cust', how="left")

In [ ]:
data = pd.merge(data, data_6_f, how='left',on='cust')

In [ ]:
# 마지막날짜부터 365일전까지 (14일씩 26번) 매주 구매건수를 계산하여 리스트에 넣음
week_to = data_2.de_dt.max()
week_trans = []
for i in range(4):
    week_from = week_to + pd.DateOffset(weeks=-1)
    week_trans.append(data_2.query('@week_from < de_dt <= @week_to')
                      .groupby('cust')['de_dt']
                      .agg([(f'w{4-i}', 'count')])
                      .reset_index())
    week_to = week_from

# 리스트로부터 데이터프레임 변환 
f = pd.DataFrame({'cust': data_2.cust.unique()})
for w in week_trans[::-1]:
    f = pd.merge(f, w, how='left')
f = f.fillna(0)
f.head()

In [ ]:
k = np.polyfit(range(4),f.iloc[:,1:].T,1)[0]

In [ ]:
data_df = pd.DataFrame(k, index=f.cust, columns=['w'])
data_df =data_df.reset_index()

In [ ]:
data = pd.merge(data,data_df,how='left')

- 연령대별 유통사 선호도에 따른 가중치

In [ ]:
data_2['age'] = pd.to_numeric(data_2['ages'].str[:2])

In [ ]:
data_2['age_group'] = data_2['age'].apply(lambda x : 'twenty' if (x>=20) & (x<30)
                                     else 'thirty' if (x>=30) & (x<40)
                                     else 'forty' if (x>=40) & (x<50)
                                     else 'fifty' if (x>=50) & (x<60)
                                     else 'sixty' if (x>=60) & (x<70)                                          
                                     else 'seventy')

twenty_prefer_brd = data_2[data_2['age_group'] == 'twenty'].cop_c.value_counts().index[1:].to_list()
thirty_prefer_brd = data_2[data_2['age_group'] == 'thirty'].cop_c.value_counts().index[1:].to_list()
forty_prefer_brd = data_2[data_2['age_group'] == 'forty'].cop_c.value_counts().index[1:].to_list()
fifty_prefer_brd = data_2[data_2['age_group'] == 'fifty'].cop_c.value_counts().index[1:].to_list()
sixty_prefer_brd = data_2[data_2['age_group'] == 'sixty'].cop_c.value_counts().index[1:].to_list()
seventy_prefer_brd = data_2[data_2['age_group'] == 'seventy'].cop_c.value_counts().index[1:].to_list()

def prefer_brd(x, list):
    for i in range(len(list)):
        if(x == list[i]):
            return len(list)-i

data_2['20_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, twenty_prefer_brd)).fillna(0)
data_2['30_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, thirty_prefer_brd)).fillna(0)
data_2['40_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, forty_prefer_brd)).fillna(0)
data_2['50_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, fifty_prefer_brd)).fillna(0)
data_2['60_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, sixty_prefer_brd)).fillna(0)
data_2['70_weight_cop'] = data_2['cop_c'].apply(lambda x: prefer_brd(x, seventy_prefer_brd)).fillna(0)

data_2['weight_sum'] = data_2['20_weight_cop'] + data_2['30_weight_cop'] + data_2['40_weight_cop'] + data_2['50_weight_cop'] + data_2['60_weight_cop'] + data_2['70_weight_cop']

In [ ]:
f = data_2.groupby('cust')['20_weight_cop', '30_weight_cop', '40_weight_cop', '50_weight_cop', '60_weight_cop', '70_weight_cop', 'weight_sum'].sum()

f['20_weight_cop'] = f['20_weight_cop'] / f['weight_sum']
f['30_weight_cop'] = f['30_weight_cop'] / f['weight_sum'] 
f['40_weight_cop'] = f['40_weight_cop'] / f['weight_sum'] 
f['50_weight_cop'] = f['50_weight_cop'] / f['weight_sum']
f['60_weight_cop'] = f['60_weight_cop'] / f['weight_sum']
f['70_weight_cop'] = f['70_weight_cop'] / f['weight_sum']
f.drop(columns='weight_sum', inplace = True)

f = f.fillna(0)
features.append(f) ;f.head()

In [ ]:
f = f.reset_index()

In [ ]:
data = pd.merge(data,f,how = 'left')

In [ ]:
train_data = list(data_2.groupby('cust')['clac_hlv_nm'].unique())
train_data

In [ ]:
def oversample(x, n):
    lst = []
    for i in x:
        tmp = []
        for j in range(n):
          random.shuffle(i)
          tmp += list(i)
        lst.append(tmp)
    return lst

In [ ]:
w2v_input = oversample(train_data, 10)

In [ ]:
w2v = word2vec.Word2Vec(sentences = w2v_input,
                        vector_size = 20, #단어 벡터의 크기
                        window = 3, #주변단어개수
                        min_count = 1, #전체 문서에 등장한 최소 개수
                        sg = 1).wv # sg : Training algorithm: 1 for skip-gram; otherwise CBOW.

In [ ]:
train_mean_vector = []
from tqdm import tqdm
for words in tqdm(train_data):
        tmp = np.zeros(20)             # 다음 customer ID에 대한 vector를 계삲하기 전 0으로 초기화
        cnt = 0
        for word in words:
            try:
                tmp += w2v[word]
                cnt += 1
            except:
                pass
        tmp /= cnt                      # customer ID 에 있는 아이템 갯수로 전체 벡터를 mean해줌  
        train_mean_vector.append(tmp)
        
train_mean_vector = np.array(train_mean_vector)

In [ ]:
pd.DataFrame(train_mean_vector).isna().sum().sum()

In [ ]:
train_mean_vector

In [ ]:
train_mean_vector = pd.DataFrame(train_mean_vector)
train_mean_vector.columns = train_mean_vector.columns.astype(str) + 'clac_hlv_nm'

In [ ]:
train_mean_vector

In [ ]:
data = pd.concat([data, train_mean_vector], axis=1)

In [ ]:
data

In [ ]:
data.to_csv("data/data_12_12.csv", index=False)